# 03 — Train configured models

**Purpose.** Fit the configured logistic-regression and XGBoost models using development admissions only.

**Training policy**
- Development data = rows labelled `train` or `validation`.
- The temporal `test` period is never used for fitting.
- Unknown or unresolved outcomes are excluded by the reusable target encoder.
- Imputation, encoding, and scaling are fitted inside each model pipeline.
- Each saved model bundle includes its feature list, target name, and training metadata.


In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
# =========================
# CELL 1 — Imports
# =========================
%load_ext autoreload
%autoreload 2

import json
import joblib
import yaml
import numpy as np
import pandas as pd

from pathlib import Path
from collections.abc import Iterable

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from icare_risk.data.audit import load_admission_file

In [2]:
# =========================
# CELL 2 — Config load
# =========================
CONFIG_PATH = Path("../../src/icare_risk/config/icare/analysis.yaml")
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

print(f"Configuration: {CONFIG_PATH}")

dataset_path = config["outputs"]["processed_dataset"]
df = load_admission_file(dataset_path)

columns_cfg = config["columns"]
target = columns_cfg["target"]

development_splits = ["train", "validation"]
development = df[df["split"].isin(development_splits)].copy()
test = df[df["split"].eq("test")].copy()

print(f"Dataset: {dataset_path}")
print(f"Rows and columns: {df.shape}")
print(f"Development splits: {development_splits}")
print(f"Development admissions: {len(development):,}")
print(f"Temporal test admissions: {len(test):,}")

Configuration: ../../src/icare_risk/config/icare/analysis.yaml
Dataset: data/processed/admissions_with_split.parquet
Rows and columns: (10000, 187)
Development splits: ['train', 'validation']
Development admissions: 7,517
Temporal test admissions: 2,483


In [3]:
# =========================
# CELL 3 — Helpers
# =========================
def _prefix_columns(columns: Iterable[str], prefixes: Iterable[str]) -> list[str]:
    return [c for c in columns if any(c.startswith(p) for p in prefixes)]

def _load_manifest(manifest_path: str | Path | None) -> dict:
    if manifest_path is None:
        raise ValueError("Feature set requires manifest_path, but none was provided.")
    path = Path(manifest_path)
    if not path.exists():
        raise FileNotFoundError(f"Manifest file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def _resolve_from_manifest(df: pd.DataFrame, definition: dict, manifest: dict) -> list[str]:
    include_groups = definition.get("include_groups", [])
    include_features = definition.get("include_features", [])

    manifest_groups = manifest.get("groups", {})
    manifest_feature_cols = manifest.get("feature_cols", [])

    selected: list[str] = []

    if include_groups:
        for group_name in include_groups:
            if group_name not in manifest_groups:
                raise KeyError(f"Group {group_name!r} not found in manifest.groups")
            selected.extend(manifest_groups[group_name])

    if include_features:
        selected.extend(include_features)

    if not include_groups and not include_features:
        selected = list(manifest_feature_cols)

    selected = list(dict.fromkeys(selected))
    missing = [c for c in selected if c not in df.columns]
    if missing:
        raise KeyError(f"Feature columns not found in dataframe: {missing}")
    return selected


def resolve_feature_set(
    df: pd.DataFrame,
    definition: dict,
    *,
    exclude_columns: Iterable[str] = (),
    manifest_path: str | Path | None = None,
) -> list[str]:
    """
    Supported definition styles in config['feature_sets'][name]:
      1) columns: [..]
      2) prefixes: [..]
      3) include_all: true
      4) from_manifest:
           include_groups: [time_cols, numeric_cols, binary_cols, categorical_cols]  # optional
           include_features: [..]                                                     # optional
    Optional:
      exclude: [..]
    """
    if not isinstance(definition, dict):
        raise TypeError("Feature-set definition must be a mapping")

    if "columns" in definition:
        selected = list(definition["columns"])
    elif "prefixes" in definition:
        selected = _prefix_columns(df.columns, definition["prefixes"])
    elif definition.get("include_all", False):
        selected = list(df.columns)
    elif "from_manifest" in definition:
        manifest = _load_manifest(manifest_path)
        selected = _resolve_from_manifest(df, definition["from_manifest"], manifest)
    else:
        raise ValueError(
            "Feature set must define one of: columns, prefixes, include_all, from_manifest"
        )

    excluded = set(exclude_columns) | set(definition.get("exclude", []))
    selected = [c for c in selected if c not in excluded]

    missing = [c for c in selected if c not in df.columns]
    if missing:
        raise KeyError(f"Feature columns not found: {missing}")
    if not selected:
        raise ValueError("Feature set resolved to zero columns")
    return selected


def validate_binary_target(y: pd.Series, target_name: str) -> pd.Series:
    if y.isna().any():
        n_missing = int(y.isna().sum())
        raise ValueError(f"Target {target_name!r} contains {n_missing} missing values.")

    # allow bool and convert
    if y.dtype == bool:
        y = y.astype(int)

    unique_vals = set(pd.Series(y).unique().tolist())
    if not unique_vals.issubset({0, 1}):
        raise ValueError(
            f"Target {target_name!r} must be binary 0/1. Found values: {sorted(unique_vals)}"
        )
    return y.astype(int)

In [4]:
# =========================
# CELL 4 — Preprocessing + model builders
# =========================
TIME_PATTERNS = ("_dslast", "_dsfirst", "days_since", "time_to_culture")


def split_feature_types(X: pd.DataFrame, feature_list: list[str]) -> dict[str, list[str]]:
    time_cols = [c for c in feature_list if any(p in c for p in TIME_PATTERNS)]

    cat_cols = [
        c for c in feature_list
        if str(X[c].dtype) in ("object", "category")
    ]

    bin_cols = []
    for c in feature_list:
        if c in time_cols or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(X[c]):
            vals = pd.Series(X[c].dropna().unique())
            if len(vals) > 0 and len(vals) <= 2 and set(vals.astype(float)).issubset({0.0, 1.0}):
                bin_cols.append(c)

    num_cols = [
        c for c in feature_list
        if c not in time_cols and c not in cat_cols and c not in bin_cols
        and pd.api.types.is_numeric_dtype(X[c])
    ]

    return {
        "time_cols": time_cols,
        "numeric_cols": num_cols,
        "binary_cols": bin_cols,
        "categorical_cols": cat_cols,
    }


def build_preprocessor(groups: dict[str, list[str]], model_type: str) -> ColumnTransformer:
    # LR: scale numeric/time. XGB: no scaling.
    scale_for_lr = model_type in {"logistic_regression", "lr"}

    time_steps = [
        ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0))]
    num_steps = [
        ("imputer", SimpleImputer(strategy="median"))]
    bin_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent"))]
    cat_steps = [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]

    if scale_for_lr:
        time_steps.append(("scaler", StandardScaler()))
        num_steps.append(("scaler", StandardScaler()))

    transformers = []

    if groups["time_cols"]:
        transformers.append(("time", Pipeline(time_steps), groups["time_cols"]))
    if groups["numeric_cols"]:
        transformers.append(("num", Pipeline(num_steps), groups["numeric_cols"]))
    if groups["binary_cols"]:
        transformers.append(("bin", Pipeline(bin_steps), groups["binary_cols"]))
    if groups["categorical_cols"]:
        transformers.append(("cat", Pipeline(cat_steps), groups["categorical_cols"]))

    if not transformers:
        raise ValueError("No transformer blocks were created. Check feature groups.")

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_estimator(model_type: str, params: dict):
    if model_type in {"logistic_regression", "lr"}:
        default_params = {
            "max_iter": 2000,
            "solver": "lbfgs",
            "n_jobs": -1,
        }
        default_params.update(params or {})
        return LogisticRegression(**default_params)

    if model_type in {"xgboost", "xgb"}:
        default_params = {
            "n_estimators": 300,
            "max_depth": 3,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": 42,
            "n_jobs": -1,
        }
        default_params.update(params or {})
        return XGBClassifier(**default_params)

    raise ValueError(f"Unsupported model_type: {model_type}")


def fit_model_pipeline(
    df_train: pd.DataFrame,
    features: list[str],
    target_col: str,
    model_type: str,
    params: dict | None = None,
):
    if target_col not in df_train.columns:
        raise KeyError(f"Target column {target_col!r} not found.")

    X = df_train[features].copy()
    y = validate_binary_target(df_train[target_col].copy(), target_col)

    groups = split_feature_types(X, features)
    preprocessor = build_preprocessor(groups, model_type=model_type)
    estimator = build_estimator(model_type=model_type, params=params or {})

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipe.fit(X, y)

    return pipe, groups

In [5]:
# =========================
# CELL 5 — Train all models from config
# =========================
if target not in development.columns:
    raise KeyError(f"Target column {target!r} is not present in the dataset.")

# Validate target once at development level
_ = validate_binary_target(development[target].copy(), target)

manifest_path = config.get("outputs", {}).get("feature_manifest_path")

models = {}
feature_map = {}
group_map = {}
training_rows = []

for model_cfg in config.get("models", []):
    name = model_cfg["name"]
    model_type = model_cfg["type"]
    feature_set_name = model_cfg["feature_set"]

    if feature_set_name not in config.get("feature_sets", {}):
        raise KeyError(f"Feature set {feature_set_name!r} is not defined in configuration.")

    definition = config["feature_sets"][feature_set_name]
    features = resolve_feature_set(
        df=df,
        definition=definition,
        exclude_columns=[target, "split"],
        manifest_path=manifest_path,
    )

    print(f"\nTraining {name}")
    print(f"  Type: {model_type}")
    print(f"  Feature set: {feature_set_name}")
    print(f"  Number of features: {len(features)}")
    print(f"  Parameters: {model_cfg.get('params', {})}")

    fitted_pipeline, groups = fit_model_pipeline(
        df_train=development,
        features=features,
        target_col=target,
        model_type=model_type,
        params=model_cfg.get("params", {}),
    )

    models[name] = fitted_pipeline
    feature_map[name] = features
    group_map[name] = groups

    y_dev = development[target]
    training_rows.append({
        "model": name,
        "model_type": model_type,
        "feature_set": feature_set_name,
        "n_features": len(features),
        "n_time": len(groups["time_cols"]),
        "n_num": len(groups["numeric_cols"]),
        "n_bin": len(groups["binary_cols"]),
        "n_cat": len(groups["categorical_cols"]),
        "n_development_rows": len(development),
        "n_test_rows": len(test),
        "n_positive_development": int((y_dev == 1).sum()),
        "n_negative_development": int((y_dev == 0).sum()),
    })

training_summary = pd.DataFrame(training_rows)
display(training_summary)


Training lr_kim_components
  Type: logistic_regression
  Feature set: kim_components
  Number of features: 5
  Parameters: {'C': 1.0}

Training lr_all_score_values
  Type: logistic_regression
  Feature set: all_score_values
  Number of features: 6
  Parameters: {'C': 1.0}

Training lr_clinical_baseline
  Type: logistic_regression
  Feature set: clinical_baseline
  Number of features: 16
  Parameters: {'C': 1.0}

Training xgb_kim_components
  Type: xgboost
  Feature set: kim_components
  Number of features: 5
  Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}

Training xgb_all_score_values
  Type: xgboost
  Feature set: all_score_values
  Number of features: 6
  Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}

Training xgb_clinical_baseline
  Type: xgboost
  Feature set: clinical_baseline
  Number of features: 16
  Parameters: {'n_estimators': 300, 'max

,model,model_type,feature_set,n_features,n_time,n_num,n_bin,n_cat,n_development_rows,n_test_rows,n_positive_development,n_negative_development
0,lr_kim_components,logistic_regression,kim_components,5,0,0,5,0,7517,2483,1485,6032
1,lr_all_score_values,logistic_regression,all_score_values,6,0,4,2,0,7517,2483,1485,6032
2,lr_clinical_baseline,logistic_regression,clinical_baseline,16,0,1,12,0,7517,2483,1485,6032
3,xgb_kim_components,xgboost,kim_components,5,0,0,5,0,7517,2483,1485,6032
4,xgb_all_score_values,xgboost,all_score_values,6,0,4,2,0,7517,2483,1485,6032
5,xgb_clinical_baseline,xgboost,clinical_baseline,16,0,1,12,0,7517,2483,1485,6032


In [6]:
# =========================
# CELL 6 — Save bundles + summary
# =========================
model_directory = Path(config["outputs"]["model_directory"])
model_directory.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    bundle = {
        "model": model,                 # full sklearn Pipeline(preprocessor+estimator)
        "features": feature_map[name],  # raw input feature list
        "feature_groups": group_map[name],
        "target": target,
        "development_splits": development_splits,
    }
    output_path = model_directory / f"{name}.joblib"
    joblib.dump(bundle, output_path)

print(f"Saved {len(models)} fitted models to {model_directory}")

summary_path = model_directory / "training_summary.csv"
training_summary.to_csv(summary_path, index=False)
print(f"Saved training summary: {summary_path}")

Saved 6 fitted models to models/fitted
Saved training summary: models/fitted/training_summary.csv
